# Qwen3-8B FineTuning — NL to CNL Translation

In [ ]:
from huggingface_hub import login
login('YOUR HUGGINGFACE_TOKEN')

In [ ]:
import os
import json
from pathlib import Path
from datasets import load_dataset
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from transformers import EarlyStoppingCallback
import trl
from trl import SFTTrainer, SFTConfig
import torch
import peft
from peft import LoraConfig

print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("Torch:", torch.__version__)


## Dataset Generation

In [ ]:
## wrap your dataset loading and splitting

def load_data(path, test_size=0.1, seed=42):
    def create_conversation(sample):
        return {
            "messages": [
                {"role": "system",
                 "content": "You are an expert in Translating the Natural language (NL) into Controlled Natural Language (CNL) translation. Always provide precise, syntactically correct translations of NL into CNL."},
                {"role": "user", "content": f"Translate the following natural language to controlled natural language: {sample['data_dict']['NL_V2']} "},
                {"role": "assistant", "content": sample["data_dict"]["CNL_V2"]},
            ]
        }


    # Load and transform
    dataset = load_dataset("json", data_files=path, split="train")
    dataset = dataset.map(create_conversation, remove_columns=dataset.features, batched=False)
    print("Dataset converted to conversational format.")

    # Split into train and test
    split_dataset = dataset.train_test_split(test_size=test_size, seed=seed)
    train_dataset = split_dataset["train"]
    test_dataset = split_dataset["test"]

    train_dataset = train_dataset.shuffle(seed=seed)

    return train_dataset, test_dataset


In [ ]:
dataset_file = "Path/To/Your/train_data.json" 
train_data, test_data = load_data(dataset_file, test_size=0.1)

# Optional: inspect an example
print("Example from train set:")
print(train_data[2])

print(f"Train dataset size: {len(train_data)}")
print(f"Test dataset size:  {len(test_data)}")

## Model Loading

In [ ]:

base_model_name = "Qwen/Qwen3-8B"

def load_prepare_model(model_name=base_model_name):
    """
    Loads and prepares the Qwen3-8B-Instruct model and tokenizer without quantization.

    Args:
        model_name (str): Hugging Face model identifier.

    Returns:
        model: Loaded causal language model.
        tokenizer: Corresponding tokenizer with padding configured.
    """
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    compute_dtype = torch.bfloat16


    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=compute_dtype,      
        device_map="auto",
        trust_remote_code=True,    
    )

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)


    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    return model, tokenizer


In [ ]:
base_model = "Qwen/Qwen3-8B" 
model, tokenizer = load_prepare_model(model_name=base_model)

## Preprocessing — Completion-Only Format

In [ ]:
def preprocess_to_completion_format(dataset, tokenizer):
    """
    Converts messages dataset into prompt+completion format compatible with
    SFTConfig(completion_only_loss=True).
    """
    # The assistant turn header in Qwen3's ChatML format
    ASSISTANT_HEADER = "<|im_start|>assistant\n"

    def convert(sample):
        # Build full conversation
        full_text = tokenizer.apply_chat_template(
            sample["messages"],      # system + user + assistant all together
            tokenize=False,
            add_generation_prompt=False,
        )

        split_idx = full_text.rfind(ASSISTANT_HEADER)
        if split_idx == -1:
            raise ValueError(
                f"Assistant header '{ASSISTANT_HEADER}' not found in rendered template.\n"
                f"Full text: {full_text[:200]}"
            )

    
        prompt     = full_text[:split_idx + len(ASSISTANT_HEADER)]
        completion = full_text[split_idx + len(ASSISTANT_HEADER):]
        return {"prompt": prompt, "completion": completion}

    return dataset.map(convert, remove_columns=dataset.column_names)


train_data_processed = preprocess_to_completion_format(train_data, tokenizer)
test_data_processed  = preprocess_to_completion_format(test_data,  tokenizer)

# Verify split is correct
sample = train_data_processed[0]
print("── PROMPT (last 100 chars) ──────────────────────────────")
print(repr(sample["prompt"][-100:]))
print("\n── COMPLETION ───────────────────────────────────────────")
print(repr(sample["completion"]))

## Trainer Setup & Training

In [ ]:
max_seq_length = 1024

def create_trainer(model, tokenizer, train_dataset, eval_dataset=None,
                   max_seq_length=max_seq_length, num_epochs=4, early_stopping_patience=2):
    """
    Creates and returns an SFTTrainer instance configured for Qwen3-8B fine-tuning.
    Dataset format expected: {"prompt": "...", "completion": "..."}

    """
    peft_config = LoraConfig(
        r=32,
        lora_alpha=64,        # 2 * r is standard
        lora_dropout=0.1,
        use_rslora=True,      # stabilizes training with rank scaling
        use_dora=False,       # mutually exclusive with use_rslora
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )

    training_arguments = SFTConfig(
        output_dir="PATH/TO/SAVE/ADAPTER",
        num_train_epochs=num_epochs,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=1,
        optim="adamw_torch",
        logging_steps=25,
        learning_rate=1e-4,
        weight_decay=0.03,
        fp16=False,
        bf16=True,
        max_grad_norm=0.3,
        max_steps=-1,
        warmup_ratio=0.05,
        group_by_length=True,
        lr_scheduler_type="cosine",
        neftune_noise_alpha=5.0,
        load_best_model_at_end=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        max_length=max_seq_length,
        completion_only_loss=True,
    )

    callbacks = []
    if eval_dataset is not None and early_stopping_patience is not None:
        callbacks.append(EarlyStoppingCallback(early_stopping_patience=early_stopping_patience))

    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        peft_config=peft_config,
        processing_class=tokenizer,
        args=training_arguments,
        callbacks=callbacks,
    )

    return trainer


In [ ]:


# Start training
trainer = create_trainer(
    model, tokenizer,
    train_dataset=train_data_processed,    # use pre-processed datasets
    eval_dataset=test_data_processed,      
    num_epochs=15,
    early_stopping_patience=3
)
trainer.train()

# Print best checkpoint path
print("Best checkpoint:", trainer.state.best_model_checkpoint)

# Save the final LoRA adapter weights and tokenizer
trainer.save_model("PATH/TO/SAVE/ADAPTER")
tokenizer.save_pretrained("PATH/TO/SAVE/ADAPTER")
print("Adapter saved to PATH/TO/SAVE/ADAPTER")
